## Setup

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams["figure.dpi"] = 120

BASE_DIR  = os.path.abspath("..")
DATA_DIR  = os.path.join(BASE_DIR, "data")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

DATA_PATH = os.path.join(DATA_DIR, "executions.csv")

## Load Raw Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}  ({df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns)")
display(df_raw.head(3))

In [ ]:
df_raw.info(verbose=True, show_counts=True)

## Null Audit

In [ ]:
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(2)
null_df     = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_df     = null_df[null_df.null_count > 0].sort_values("null_count", ascending=False)

if null_df.empty:
    print("No nulls found in the dataset.")
else:
    display(null_df)
    print(f"\nTotal null cells: {null_counts.sum():,}")

## Feature Drop Decisions

#### Zero-Variance Columns

Columns that carry a single value across all 9,400 rows contribute zero information.

In [ ]:
# Expected zero-variance cols per Feature Decision Sheet
ZERO_VARIANCE_DROPS = ["grade", "uses_lists", "uses_dicts",
                       "uses_recursion", "uses_exceptions", "inside_try"]

print("=== Empirical zero-variance verification ===")
for col in ZERO_VARIANCE_DROPS:
    uniq = df_raw[col].unique()
    status = "CONFIRMED" if len(uniq) == 1 else f"✗ has {len(uniq)} values — REVIEW"
    print(f"  {col:<30} unique={list(uniq)}  {status}")

#### Exact Duplicates / Near-Redundant Columns

In [ ]:
REDUNDANT_DROPS = ["is_inside_loop", "total_time_ms_at_line",
                   "min_time_ms_at_line", "max_time_ms_at_line"]

# is_inside_loop vs inside_loop
overlap = (df_raw["is_inside_loop"] == df_raw["inside_loop"]).mean()
print(f"is_inside_loop == inside_loop: {overlap:.4f}  {'DUPLICATE' if overlap == 1.0 else 'NOT IDENTICAL'}")

# total_time vs derived product
derived = df_raw["execution_count_at_line"] * df_raw["avg_time_ms_at_line"]
corr_total = df_raw["total_time_ms_at_line"].corr(derived)
print(f"total_time_ms_at_line corr with exec_countxavg_time: {corr_total:.4f}")

# min/max vs avg
corr_min = df_raw["min_time_ms_at_line"].corr(df_raw["avg_time_ms_at_line"])
corr_max = df_raw["max_time_ms_at_line"].corr(df_raw["avg_time_ms_at_line"])
print(f"min_time_ms corr with avg_time_ms: {corr_min:.4f}")
print(f"max_time_ms corr with avg_time_ms: {corr_max:.4f}")

#### Target Leakage

In [ ]:
LEAKAGE_DROPS = ["score", "correctness_score", "efficiency_complexity_score",
                 "quality_score", "maintainability_score"]
print("Leakage columns (excluded from clustering and classification features):")
for col in LEAKAGE_DROPS:
    exists = col in df_raw.columns
    print(f"  {col:<35} in dataset: {exists}")

display(df_raw[["impact_score"]].describe().T)

#### Program-Level Noise

In [ ]:
PROGRAM_LEVEL_DROPS = [
    "token_count", "ast_node_count", "source_lines", "execution_time_ms",
    "total_lines_executed", "unique_lines_profiled", "peak_memory_bytes",
    "total_suggestions", "suggestion_density", "complexity_confidence",
    "function_count", "loop_count", "if_count", "try_count",
    "assignment_count", "call_count", "binary_op_count",
    "high_severity_count", "medium_severity_count", "low_severity_count",
    "static_suggestion_count", "hybrid_suggestion_count", "dynamic_suggestion_count",
    "count_unused_vars", "count_dead_code", "count_constant_folding",
    "count_early_return", "count_loop_invariant", "count_string_concat_loop",
    "count_nested_loops", "count_hot_loop", "count_repeated_computation",
    "count_expensive_calls",
]

present = [c for c in PROGRAM_LEVEL_DROPS if c in df_raw.columns]
print(f"Program-level columns to drop: {len(present)} of {len(PROGRAM_LEVEL_DROPS)} present")

sample_prog_cols = [c for c in ["token_count", "source_lines", "total_suggestions"] if c in df_raw.columns]
if sample_prog_cols:
    print("\nSample program-level columns — note they repeat per program:")
    display(df_raw[sample_prog_cols].describe().T)

#### Identifiers & Non-Generalizable Columns

In [ ]:
IDENTIFIER_DROPS = ["line_number", "nearest_function_name",
                    "same_line_suggestion_count", "co_occurring_patterns"]

# Verify near-zero variance of same_line_suggestion_count
slsc = df_raw["same_line_suggestion_count"].value_counts(normalize=True)
print("same_line_suggestion_count distribution:")
print(slsc.to_string())
print(f"\n{slsc.iloc[0]*100:.1f}% of rows have value=1 (near-zero variance → drop)")

# Verify nearest_function_name — only 4 dataset-specific values
print("\nnearest_function_name unique values:")
print(df_raw["nearest_function_name"].value_counts(dropna=False))

#### Circular / Derived Columns

In [ ]:
CIRCULAR_DROPS = ["detector_family", "score_dimension",
                  "complexity_class", "severity", "pattern"]

print("detector_family unique:",   df_raw["detector_family"].unique())
print("score_dimension unique:",   df_raw["score_dimension"].unique())
print("complexity_class unique:",  df_raw["complexity_class"].unique())
print("severity unique:",          df_raw["severity"].unique())
print("\npattern value counts:")
display(df_raw["pattern"].value_counts().to_frame())

#### Drop Summary

In [ ]:
ALL_DROPS = list(set(
    ZERO_VARIANCE_DROPS
    + REDUNDANT_DROPS
    + LEAKAGE_DROPS
    + PROGRAM_LEVEL_DROPS
    + IDENTIFIER_DROPS
    + CIRCULAR_DROPS
))

drops_present = [c for c in ALL_DROPS if c in df_raw.columns]
drops_missing = [c for c in ALL_DROPS if c not in df_raw.columns]

print(f"Total columns in raw data : {df_raw.shape[1]}")
print(f"Columns scheduled to drop  : {len(drops_present)}")
print(f"Remaining after drop       : {df_raw.shape[1] - len(drops_present)}")
if drops_missing:
    print(f"\nColumns not found in dataset (already absent or renamed): {drops_missing}")

## Apply Drops → Working DataFrame

In [ ]:
df = df_raw.drop(columns=drops_present, errors="ignore").copy()
print(f"Shape after drops: {df.shape}")
display(df.dtypes.to_frame("dtype").T)

## Final Feature Matrix: Keep Columns

In [ ]:
# Structural context 
STRUCT_COLS = [
    "severity_ordinal", "loop_depth", "branch_depth", "function_depth",
    "inside_function", "inside_loop", "inside_branch", "max_recursion_depth",
    "complexity_ordinal", "relative_line_position",
]

# Dynamic / runtime
DYNAMIC_COLS = [
    "execution_count_at_line", "avg_time_ms_at_line", "line_dominance",
    "line_execution_rank", "line_time_rank", "memory_bytes_at_line",
]

# Function-level dynamic (3)
FUNC_COLS = [
    "function_call_count", "function_total_time_ms", "function_avg_time_ms",
]

# Categorical to one-hot (1 → 4 binary)
CAT_COLS = ["node_type_at_line"]

FEATURE_COLS = STRUCT_COLS + DYNAMIC_COLS + FUNC_COLS + CAT_COLS

missing_feats = [c for c in FEATURE_COLS if c not in df.columns]
print(f"Feature columns defined  : {len(FEATURE_COLS)}")
print(f"Missing from working df  : {missing_feats or 'None'}")
print(f"Final feature count (pre-OHE): {len(FEATURE_COLS)} → post-OHE: ~{len(FEATURE_COLS) - 1 + 4}")

## Descriptive Statistics — Retained Features

#### Structural Features

In [ ]:
display(df[STRUCT_COLS].describe().T.round(4))

**Key observations:**
- `severity_ordinal` is heavily skewed toward 1 (low): 87.2% of rows.
- `complexity_ordinal` has only two values — 3 and 5 (1,800 vs 7,600 rows).
- `branch_depth` is binary (0/1); `max_recursion_depth` is binary (0/1).
- `relative_line_position` is bimodal — most suggestions cluster near the start (0.29) or end (0.92) of functions.

#### Dynamic / Runtime Features

In [ ]:
display(df[DYNAMIC_COLS].describe().T.round(6))

print("\nZero-proportion in sparse runtime columns:")
for c in ["execution_count_at_line", "avg_time_ms_at_line", "memory_bytes_at_line"]:
    z = (df[c] == 0).mean()
    print(f"  {c:<35}: {z*100:.1f}% zeros")

**Key observations:**
- `execution_count_at_line` and `avg_time_ms_at_line` have **36.2% zeros** — static suggestions that were never executed. These will use `log1p` transform before scaling.
- `execution_count_at_line` takes only 6 discrete values: {0, 1, 5, 8, 13, 30}.
- `memory_bytes_at_line` is right-skewed with a narrow range (2% zeros).

#### Function-Level Dynamic Features

In [ ]:
display(df[FUNC_COLS].describe().T.round(6))

print("\nSkewness (before log1p):")
for c in FUNC_COLS:
    print(f"  {c:<35}: {df[c].skew():.2f}")

**Key observations:**
- All three function-level columns are **heavily right-skewed** (skew > 5.9) — dominated by zero values for static suggestions. `log1p` transform is mandatory before StandardScaler.

#### Categorical Feature — `node_type_at_line`

In [ ]:
node_counts = df["node_type_at_line"].value_counts()
node_pct    = (node_counts / len(df) * 100).round(1)
display(pd.DataFrame({"count": node_counts, "pct": node_pct}))
print(f"\nCardinality: {df['node_type_at_line'].nunique()} — will produce 4 one-hot columns")

## Distribution Analysis

#### Pattern Frequency

In [ ]:
pat_counts = df_raw["pattern"].value_counts()
pat_pct    = (pat_counts / len(df_raw) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = pat_counts.sort_values(ascending=True).plot(
    kind="barh", ax=ax, color="#4C72B0", edgecolor="white", linewidth=0.5
)
for bar, pct in zip(ax.patches, pat_pct.sort_values(ascending=True)):
    ax.text(
        bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
        f"{pct}%", va="center", fontsize=9
    )
ax.set_xlabel("Row count")
ax.set_title("Pattern distribution (n=9,400)", fontsize=13)
ax.set_xlim(0, ax.get_xlim()[1] * 1.12)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "pattern_frequency.png"), dpi=150)
plt.show()

#### Severity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Severity count
sev_counts = df_raw["severity"].value_counts().reindex(["low", "medium", "high"])
colors = ["#4C72B0", "#DD8452", "#C44E52"]
sev_counts.plot(kind="bar", ax=axes[0], color=colors, edgecolor="white", rot=0)
axes[0].set_title("Severity count", fontsize=12)
axes[0].set_ylabel("Count")
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
        f"{bar.get_height():.0f}", ha="center", fontsize=9
    )

# Pattern × Severity heatmap
cross = pd.crosstab(df_raw["pattern"], df_raw["severity"])[["low", "medium", "high"]]
sns.heatmap(cross, ax=axes[1], annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, linecolor="white", cbar=False)
axes[1].set_title("Pattern × Severity heatmap", fontsize=12)
axes[1].set_xlabel("Severity")
axes[1].set_ylabel("Pattern")

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "severity_distribution.png"), dpi=150)
plt.show()

print("\nObservation: severity maps perfectly onto pattern — confirming severity is circular for clustering.")

#### Detector Family × Pattern

In [ ]:
cross_det = pd.crosstab(df_raw["pattern"], df_raw["detector_family"])
display(cross_det)
print("\n→ Each row has exactly one non-zero column: 1:1 mapping confirmed. Drop detector_family from features.")

#### Runtime Feature Distributions (log scale)

Showing pre- and post-`log1p` to motivate the transform.

In [ ]:
log1p_cols = [
    "execution_count_at_line", "avg_time_ms_at_line",
    "function_call_count", "function_total_time_ms",
]

fig, axes = plt.subplots(2, len(log1p_cols), figsize=(16, 6))
for j, col in enumerate(log1p_cols):
    # Raw
    axes[0, j].hist(df[col], bins=40, color="#4C72B0", edgecolor="white", alpha=0.85)
    axes[0, j].set_title(f"{col}\n(raw)", fontsize=9)
    axes[0, j].set_ylabel("Count" if j == 0 else "")
    # log1p
    axes[1, j].hist(np.log1p(df[col]), bins=40, color="#55A868", edgecolor="white", alpha=0.85)
    axes[1, j].set_title(f"{col}\n(log1p)", fontsize=9)
    axes[1, j].set_ylabel("Count" if j == 0 else "")

fig.suptitle("Skewed runtime columns — raw vs log1p transform", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "runtime_log1p_transforms.png"), dpi=150, bbox_inches="tight")
plt.show()

#### Structural Discrete Features

In [ ]:
discrete_cols = ["loop_depth", "branch_depth", "function_depth",
                 "max_recursion_depth", "severity_ordinal", "complexity_ordinal"]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(discrete_cols):
    vc = df[col].value_counts().sort_index()
    axes[i].bar(vc.index.astype(str), vc.values, color="#4C72B0", edgecolor="white")
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel("Count")
    for bar in axes[i].patches:
        axes[i].text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{bar.get_height():.0f}", ha="center", fontsize=8
        )

plt.suptitle("Structural discrete feature distributions", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "structural_discrete_distributions.png"), dpi=150)
plt.show()

#### `impact_score`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_raw["impact_score"], bins=30, color="#C44E52", edgecolor="white", alpha=0.85)
axes[0].set_title("impact_score distribution", fontsize=12)
axes[0].set_xlabel("impact_score")
axes[0].set_ylabel("Count")

impact_by_pattern = df_raw.groupby("pattern")["impact_score"].median().sort_values(ascending=False)
impact_by_pattern.plot(kind="bar", ax=axes[1], color="#C44E52", edgecolor="white", rot=45)
axes[1].set_title("Median impact_score by pattern", fontsize=12)
axes[1].set_ylabel("Median impact_score")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "impact_score_distribution.png"), dpi=150)
plt.show()

print(f"impact_score range: [{df_raw.impact_score.min()}, {df_raw.impact_score.max()}]")
print(f"Mean: {df_raw.impact_score.mean():.2f}, Median: {df_raw.impact_score.median():.2f}, Std: {df_raw.impact_score.std():.2f}")

### Correlation Analysis — Retained Numeric Features

In [ ]:
numeric_keep = [c for c in STRUCT_COLS + DYNAMIC_COLS + FUNC_COLS if c in df.columns]
corr = df[numeric_keep].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(15, 11))
sns.heatmap(
    corr, mask=mask, ax=ax,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    linewidths=0.3, linecolor="white",
    cbar_kws={"shrink": 0.7},
)
ax.set_title("Correlation matrix — retained numeric features", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "correlation_heatmap_keep_cols.png"), dpi=150)
plt.show()

# Flag high correlations
high_corr = (
    corr.where(mask == False).abs()
    .stack()
    .reset_index()
    .rename(columns={0: "corr", "level_0": "col_a", "level_1": "col_b"})
    .query("corr >= 0.7 and col_a != col_b")
    .sort_values("corr", ascending=False)
)
if not high_corr.empty:
    print("High-correlation pairs (|r| ≥ 0.70) among retained features:")
    display(high_corr)
else:
    print("No high-correlation pairs found among retained features — feature set is clean.")

### Preprocessing 

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import warnings

df_feat = df[FEATURE_COLS].copy()

# 1. log1p on sparse/skewed runtime columns
LOG1P_COLS = [
    "execution_count_at_line", "avg_time_ms_at_line",
    "function_call_count", "function_total_time_ms", "function_avg_time_ms",
]
for c in LOG1P_COLS:
    df_feat[c] = np.log1p(df_feat[c])

# 2. One-hot encode node_type_at_line
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
node_encoded = ohe.fit_transform(df_feat[["node_type_at_line"]])
node_cols    = [f"node_{c}" for c in ohe.categories_[0]]
node_df      = pd.DataFrame(node_encoded, columns=node_cols, index=df_feat.index)
df_feat      = df_feat.drop(columns="node_type_at_line")
df_feat      = pd.concat([df_feat, node_df], axis=1)

# 3. Convert bool columns to int
bool_cols = ["inside_function", "inside_loop", "inside_branch"]
for c in bool_cols:
    if c in df_feat.columns:
        df_feat[c] = df_feat[c].astype(int)

# 4. StandardScaler on all numeric columns
scaler       = StandardScaler()
df_feat_arr  = scaler.fit_transform(df_feat)
df_feat_scaled = pd.DataFrame(df_feat_arr, columns=df_feat.columns, index=df_feat.index)

print(f"Final clustering feature matrix shape: {df_feat_scaled.shape}")
print(f"Columns ({df_feat_scaled.shape[1]}): {list(df_feat_scaled.columns)}")

In [ ]:
# Verify no nulls after preprocessing
nulls = df_feat_scaled.isnull().sum().sum()
assert nulls == 0, f"Nulls in feature matrix: {nulls}"
print(f"✓ No nulls in scaled feature matrix ({df_feat_scaled.shape[0]:,} rows × {df_feat_scaled.shape[1]} features)")

# Verify all columns near zero mean after scaling
mean_check = df_feat_scaled.mean().abs().max()
std_check  = df_feat_scaled.std().max()
print(f"✓ Max absolute mean after scaling: {mean_check:.4f}  (expected ≈ 0)")
print(f"✓ Max std after scaling           : {std_check:.4f}  (expected ≈ 1)")

## Save Outputs

In [ ]:
# Clean working df 
clean_path = os.path.join(DATA_DIR, "executions_clean.csv")
df[FEATURE_COLS].to_csv(clean_path, index=False)
print(f"Saved clean feature df")

# Scaled feature matrix for clustering 
scaled_path = os.path.join(DATA_DIR, "executions_features_scaled.csv")
df_feat_scaled.to_csv(scaled_path, index=False)
print(f"Saved scaled feature matrix")

# Metadata columns for post-cluster validation
META_COLS = ["pattern", "severity", "detector_family", "score_dimension",
             "impact_score", "co_occurring_patterns"]
meta_present = [c for c in META_COLS if c in df_raw.columns]
meta_path = os.path.join(DATA_DIR, "executions_meta.csv")
df_raw[meta_present].to_csv(meta_path, index=False)
print(f"Saved metadata df")

## EDA Summary

- Total raw columns: 79 
- Columns dropped: 56 (zero-variance, duplicates, leakage, program-level, identifiers, circular) 
- Final feature columns: 23 (pre-OHE) → **26** (post-OHE of `node_type_at_line` → 4 binary) 
- Rows: 9,400 suggestion-level rows — no missing values in retained features 
- Dominant pattern: `constant_folding` (51%) + `early_return` (25.5%) — expected class imbalance 
- Sparse runtime features: 36.2% zeros in `execution_count_at_line` / `avg_time_ms_at_line` — use `log1p` 
- Function-level features: Heavy right skew (skew > 5.9) — use `log1p` 
- Redundancy confirmed: `is_inside_loop` == `inside_loop` (100%); `total_time_ms` corr 0.996 with derived product 
- Leakage confirmed: `severity` maps 1:1 onto `pattern`; `detector_family` maps 1:1 onto `pattern` 
- Ready for Task 1: `executions_features_scaled.csv` (26 cols) 
- Ready for Task 3: `impact_score` in `executions_meta.csv` (range 2–18, mean 3.5) 

In [ ]:
print("EDA completed")